# Test notebook

The purpose of this notebook is to test an equation and compare them with the baselines: Burton, MBR, and DDM1, 2 and 3. We will also plot each storm and get the metrics for the equation.

The only cell that we have to modify is the following one, where we can change the features, the mode (template or default) and the output directory for the plots.
Raw EQ is the equation that we want to test, the raw version generated from the train_script.py file.

In [1]:
%cd /mnt/data/symbolic-regression-dst-public-repo

/mnt/data/symbolic-regression-dst-public-repo


In [ ]:
import os

FEATURES = ["Vp", "Np", "Bzsouth", "Bmag", "DST"]
MODE = "template"  # 'template' or 'default'
OUTPUT_DIR = "template_primitive_features"
RAW_EQS = [
    "((Vp * ((Bzsouth + 1.4056387) * ((DST * (0.0010231906 / (-1.9690949 - Np))) - 0.04994203))) - (DST - (Np + 26.93975))) / (51.730183 / sqrt(sqrt(Np * Bmag)))",
    "(DST * -0.052771024) - ((((Vp + (Bzsouth * Np)) + (DST / 0.44165215)) * 0.0024323284) * Bzsouth)",
    "((DST - Np) * -0.04895255) + (Bzsouth * ((((DST * 2.4618247) + Vp) + (Bzsouth * Np)) * -0.0025749283))",
]

output_folder = OUTPUT_DIR
# Count number of existing subfolders
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from tqdm import tqdm

from sympy.printing import latex

# Internal module imports
import storm_dates
import baseline_models

# from evaluation_engine import UnifiedModel, simulate_storm, compute_features
from evaluation_engine import EquationModel, simulate_storm
from train_script import load_and_preprocess, compute_features

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


In [4]:
raw_data = load_and_preprocess()
data = compute_features(raw_data)

Reading from file ./data/all_timeline/ace_imf_1h_1998.csv
Reading from file ./data/all_timeline/ace_imf_1h_1999.csv
Reading from file ./data/all_timeline/ace_imf_1h_2000.csv
Reading from file ./data/all_timeline/ace_imf_1h_2001.csv
Reading from file ./data/all_timeline/ace_imf_1h_2002.csv
Reading from file ./data/all_timeline/ace_imf_1h_2003.csv
Reading from file ./data/all_timeline/ace_imf_1h_2004.csv
Reading from file ./data/all_timeline/ace_imf_1h_2005.csv
Reading from file ./data/all_timeline/ace_imf_1h_2006.csv
Reading from file ./data/all_timeline/ace_imf_1h_2007.csv
Reading from file ./data/all_timeline/ace_imf_1h_2008.csv
Reading from file ./data/all_timeline/ace_imf_1h_2009.csv
Reading from file ./data/all_timeline/ace_imf_1h_2010.csv
Reading from file ./data/all_timeline/ace_imf_1h_2011.csv
Reading from file ./data/all_timeline/ace_imf_1h_2012.csv
Reading from file ./data/all_timeline/ace_imf_1h_2013.csv
Reading from file ./data/all_timeline/ace_imf_1h_2014.csv
Reading from f

In [5]:
def predict_and_plot_storm(model, eqs, start, end, storm_df, storm_id, save_path):
    # 1. Generate Predictions
    y_true = storm_df[start:end]["DST"].values
    colors = ["blue", "yellow", "green", "orange", "purple", "cyan"]
    res_eqs = []
    string_title = f"Storm {storm_id} Reconstruction\n"
    metrics_info = []
    
    
    for eq_index, eq in enumerate(eqs):
        res_eq = simulate_storm(model[eq], storm_df)
        res_eq = res_eq[start:end]["DST_pred"].values
        m_eq = baseline_models.get_all_metrics_dict(y_true, res_eq)
        metrics_info.append(m_eq)
        string_title += f"Evaluation for Equation {eq_index + 1} ({colors[eq_index]}): ${model[eq].latex_str()}$ \n"
    
        res_eqs.append(res_eq)

    # 2. Calculate Metrics
    
    # 3. Setup Figure (3 Columns)
    fig, axs = plt.subplots(1, 3, figsize=(24, 7), constrained_layout=True)
    fig.suptitle(
        string_title, fontsize=16
    )
    # Column 1: Time Series
    axs[0].plot(
        storm_df[start:end].index,
        y_true,
        color="black",
        label="Observed",
        alpha=0.6,
        linewidth=2,
    )
    
    for eq_index, res_eq in enumerate(res_eqs):
    
        axs[0].plot(
            storm_df[start:end].index,
            res_eqs[eq_index],
            color=colors[eq_index],
            linestyle="--",
            label=f"Equation {eq_index + 1}",
            linewidth=1.5,
        )
    
    
    axs[0].legend()
    axs[0].grid(True)
    axs[0].set_xlim(start, end)
    
    diffs = []
    
    for eq_index, res_eq in enumerate(res_eqs):
        diff_eq = res_eq - y_true
        diffs.append(diff_eq)

        axs[1].plot(storm_df[start:end].index, diff_eq, color=colors[eq_index], label=f"Equation {eq_index + 1} Error")
    
    axs[1].axhline(0, color="black", linestyle="--")

    title_metrics = (
        f"Error Comparison\n"       
    )
    
    for eq_index, m_eq in enumerate(metrics_info):
        title_metrics += f"Eq {eq_index + 1}: MAE={m_eq['MAE']:.2f}, RMSE={m_eq['RMSE']:.2f}, R²={m_eq['R2']:.3f}, BFE={m_eq['BFE']:.3f}\n"
        
        
    axs[1].set_title(title_metrics, fontsize=14)
    axs[1].set_ylabel("Error (nT)")
    axs[1].legend()
    axs[1].grid(True)
    axs[1].set_xlim(start, end)

    # Column 3: BFE
    baseline_models.plot_evaluation_bfe_multi(
        axs[2],
        y_true,
        res_eqs,
        [f"Equation {i+1}" for i in range(len(eqs))],
        [colors[i] for i in range(len(eqs))],        
    )

    plt.savefig(save_path)
    plt.close()


In [6]:
def save_prediction_data(model, eqs, start, end, storm_df, output_path):
    """
    Generates and saves a CSV with observed and predicted DST and dDST/dt.
    """
    # 1. Observed Data
    # Real dDST is calculated as the difference to the next hour
    real_dst = storm_df[start:end]["DST"].values
    real_ddst = storm_df[start:end]["DST"].diff().shift(-1).values

    # 2. Equation Predictions
    # We need the iterative predictions for DST
    pred_dst_eqs = []
    for eq_index, eq in enumerate(eqs):
        pred_dst_eq = simulate_storm(model[eq], storm_df)
        if model[eq].is_template:
            pred_dst_eq = pred_dst_eq[start:end][
                ["DST_pred", "dDST", "injection_component", "decay_component"]
            ]
        else:
            pred_dst_eq = pred_dst_eq[start:end][["DST_pred", "dDST"]]
        pred_dst_eqs.append(pred_dst_eq)
        
    # 3. Baseline Predictions (Burton & OBM)
    
    # 4. Construct Comprehensive DataFrame

    if model[eq].is_template:
        results_df = pd.DataFrame(
            {
                "Timestamp": storm_df[start:end].index,
                "Observed_DST": real_dst,
                "Real_dDST_dt": real_ddst,
                "Pred_DST_Equation": pred_dst_eq["DST_pred"].values,
                "Pred_dDST_dt_Equation": pred_dst_eq["dDST"].values,
                "Injection_Component": pred_dst_eq["injection_component"].values,
                "Decay_Component": pred_dst_eq["decay_component"].values,
                
            }
        ).set_index("Timestamp")
        
        for eq_index, pred_dst_eq in enumerate(pred_dst_eqs):
            results_df[f"Pred_DST_Equation_{eq_index+1}"] = pred_dst_eq["DST_pred"].values
            results_df[f"Pred_dDST_dt_Equation_{eq_index+1}"] = pred_dst_eq["dDST"].values
            results_df[f"Injection_Component_{eq_index+1}"] = pred_dst_eq["injection_component"].values
            results_df[f"Decay_Component_{eq_index+1}"] = pred_dst_eq["decay_component"].values
        
    else:
        results_df = pd.DataFrame(
            {
                "Timestamp": storm_df[start:end].index,
                "Observed_DST": real_dst,
                "Real_dDST_dt": real_ddst,                
            }
        ).set_index("Timestamp")
        
        for eq_index, pred_dst_eq in enumerate(pred_dst_eqs):
            results_df[f"Pred_DST_Equation_{eq_index+1}"] = pred_dst_eq["DST_pred"].values
            results_df[f"Pred_dDST_dt_Equation_{eq_index+1}"] = pred_dst_eq["dDST"].values


    results_df.to_csv(output_path)
    return results_df

## Test storms

In [7]:
storms = []
storm_indices = []
models = {}

for RAW_EQ in RAW_EQS:
    model = EquationModel(RAW_EQ, FEATURES, is_template=MODE == "template")
    models[RAW_EQ] = model
    
    
test_storms = storm_dates.TEST_STORMS_SYMBOLIC_REGRESSION

for sd, ed, storm_id in tqdm(test_storms):
    start = pd.to_datetime(sd)
    end = pd.to_datetime(ed)
    storm_df = data[
        start - pd.DateOffset(hours=1) : end + pd.DateOffset(hours=1)
    ].copy()
    if storm_df.empty:
        continue

    file_name = f"storm_{storm_id}.png"
    predict_and_plot_storm(
        models,
        RAW_EQS,
        start,
        end,
        storm_df,
        storm_id,
        os.path.join(OUTPUT_DIR, file_name),
    )

    csv_name = f"data_storm_{storm_id}.csv"
    storms.append(
        save_prediction_data(
            models, RAW_EQS, start, end, storm_df, os.path.join(OUTPUT_DIR, csv_name)
        )
    )
    storm_indices.append(storm_id)
    
with open(os.path.join(OUTPUT_DIR, 'equation.txt'), 'a') as f:
    f.write(f'Equation: {RAW_EQ}\n')            
    f.write(f'LaTeX: {latex(models[RAW_EQ].latex_str())}\n')

  0%|          | 0/20 [00:00<?, ?it/s]

100%|██████████| 20/20 [00:09<00:00,  2.10it/s]


In [8]:
storms[0].columns

Index(['Observed_DST', 'Real_dDST_dt', 'Pred_DST_Equation',
       'Pred_dDST_dt_Equation', 'Injection_Component', 'Decay_Component',
       'Pred_DST_Equation_1', 'Pred_dDST_dt_Equation_1',
       'Injection_Component_1', 'Decay_Component_1'],
      dtype='object')

In [9]:
metrics = ["RMSE", "MAE", "R2", "CC", "BFE"]
equations = [f"Equation {i+1}" for i in range(len(RAW_EQS))]

columns = [f"{eq}_{metric}" for eq in equations for metric in metrics]

summary_df = pd.DataFrame(
    columns=["Storm Index"] + columns,
)

for storm_index, storm in enumerate(storms):
    summary_df.loc[storm_indices[storm_index], "Storm Index"] = storm_indices[storm_index]


for storm_index, storm in enumerate(storms):
    y_true = storm["Observed_DST"].values
    
    for eq_index in range(len(RAW_EQS)):
    
        res_eq = storm[f"Pred_DST_Equation_{eq_index+1}"].values
    

        m_eq = baseline_models.get_all_metrics_dict(y_true, res_eq)
        
        for metric in metrics:
            summary_df.loc[storm_indices[storm_index], f"Equation {eq_index+1}_{metric}"] = m_eq[metric]
        
        

summary_df.loc[len(summary_df)] = ["Mean", *summary_df[columns].mean().values]


global_data = pd.concat(storms, ignore_index=True)
y_true = global_data["Observed_DST"].values

summary_df.loc[len(summary_df), "Storm Index"] = 'Global'

for eq_index in range(len(RAW_EQS)):
    res_eq = global_data[f"Pred_DST_Equation_{eq_index+1}"].values
    m_eq = baseline_models.get_all_metrics_dict(y_true, res_eq)    
    for metric in metrics:
        summary_df.loc[len(summary_df) - 1, f"Equation {eq_index+1}_{metric}"] = m_eq[metric]


display(summary_df)

,Storm Index,Equation 1_RMSE,Equation 1_MAE,Equation 1_R2,Equation 1_CC,Equation 1_BFE
54,54,9.745655,6.83855,0.765586,0.923463,12.618396
55,55,15.45666,12.125151,0.802554,0.914472,17.293221
56,56,11.923871,9.441087,0.687895,0.864996,13.776978
57,57,10.548566,8.06729,0.717111,0.885463,10.447585
58,58,9.316351,6.358408,0.790282,0.915747,15.773875
59,59,19.212159,16.20118,0.628309,0.910652,12.301995
60,60,16.652461,13.110027,0.801415,0.960356,20.074287
61,61,11.789867,8.40107,0.917132,0.961177,17.16978
62,62,12.65854,10.555086,0.627468,0.801518,14.578574
63,63,13.332099,8.723691,0.870862,0.962578,20.154227


In [10]:
print(summary_df.loc[summary_df["Storm Index"].isin([57, 68, 'Mean'])].to_latex(index=False, float_format="%.3f"))

\begin{tabular}{llllll}
\toprule
Storm Index & Equation 1_RMSE & Equation 1_MAE & Equation 1_R2 & Equation 1_CC & Equation 1_BFE \\
\midrule
57 & 10.549 & 8.067 & 0.717 & 0.885 & 10.448 \\
68 & 18.717 & 15.040 & 0.976 & 0.988 & 17.226 \\
Mean & 13.523 & 9.956 & 0.794 & 0.918 & 15.843 \\
\bottomrule
\end{tabular}



In [ ]:
print(summary_df.to_latex(index=False, float_format="%.3f").replace("_", " ").replace("Equation ", "Eq ").replace("Storm Index", "Storm"))

NameError: name 'xxxx' is not defined

In [ ]:
display(summary_df[['Storm Index', 'Equation 1_BFE', 'Equation 2_BFE', 'Equation 3_BFE']].set_index('Storm Index'))

## Train storms

In [ ]:
storms = []
storm_indices = []
models = {}

for RAW_EQ in RAW_EQS:
    model = EquationModel(RAW_EQ, FEATURES, is_template=MODE == "template")
    models[RAW_EQ] = model
    
    
test_storms = storm_dates.TRAIN_STORMS_SYMBOLIC_REGRESSION

for sd, ed, storm_id in tqdm(test_storms):
    start = pd.to_datetime(sd)
    end = pd.to_datetime(ed)
    storm_df = data[
        start - pd.DateOffset(hours=1) : end + pd.DateOffset(hours=1)
    ].copy()
    if storm_df.empty:
        continue

    file_name = f"storm_{storm_id}.png"
    predict_and_plot_storm(
        models,
        RAW_EQS,
        start,
        end,
        storm_df,
        storm_id,
        os.path.join(OUTPUT_DIR, file_name),
    )

    csv_name = f"data_storm_{storm_id}.csv"
    storms.append(
        save_prediction_data(
            models, RAW_EQS, start, end, storm_df, os.path.join(OUTPUT_DIR, csv_name)
        )
    )
    storm_indices.append(storm_id)


In [ ]:
metrics = ["RMSE", "MAE", "R2", "CC", "BFE"]
equations = [f"Equation {i+1}" for i in range(len(RAW_EQS))]

columns = [f"{eq}_{metric}" for eq in equations for metric in metrics]

summary_df = pd.DataFrame(
    columns=["Storm Index"] + columns,
)

for storm_index, storm in enumerate(storms):
    summary_df.loc[len(summary_df), "Storm Index"] = storm_indices[storm_index]


for storm_index, storm in enumerate(storms):
    y_true = storm["Observed_DST"].values
    
    for eq_index in range(len(RAW_EQS)):
    
        res_eq = storm[f"Pred_DST_Equation_{eq_index+1}"].values
    

        m_eq = baseline_models.get_all_metrics_dict(y_true, res_eq)
        
        for metric in metrics:
            summary_df.loc[storm_indices[storm_index], f"Equation {eq_index+1}_{metric}"] = m_eq[metric]
        
        

display(summary_df.mean())

summary_df.loc[len(summary_df)] = ["Mean", *summary_df[columns].mean().values]

global_data = pd.concat(storms, ignore_index=True)
y_true = global_data["Observed_DST"].values

summary_df.loc[len(summary_df), "Storm Index"] = 'Global'



for eq_index in range(len(RAW_EQS)):
    res_eq = global_data[f"Pred_DST_Equation_{eq_index+1}"].values
    m_eq = baseline_models.get_all_metrics_dict(y_true, res_eq)    
    for metric in metrics:
        summary_df.loc[len(summary_df) - 1, f"Equation {eq_index+1}_{metric}"] = m_eq[metric]


display(summary_df)

In [ ]:
display(summary_df[['Storm Index', 'Equation 1_BFE', 'Equation 2_BFE', 'Equation 3_BFE']].set_index('Storm Index'))

In [ ]:

storms = range(54, 74)
parent_folder = 'unconstrained-derived-figures'

for storm_number in storms:
    # We use f-strings with double {{ }} to escape the LaTeX braces
    # and single { } for the Python variables.
    latex_code = f"""
\\begin{{figure}}[ht]
    \\centering
    \\includegraphics[width=\\textwidth]{{{parent_folder}/storm_{storm_number}.png}}
    \\caption{{Reconstruction of storm {storm_number} using the Equations generated from the unconstrained symbolic regression with the derived features}}\\label{{fig:unconstrained-storm-{storm_number}}}
\\end{{figure}}
"""
    print(latex_code)